In [ ]:
import lightning as L
import torch
from IPython.display import Video

from training.evaluate import evaluate, record_video
from training.ppo import ActorCritic, PolicyModule, RolloutData, flat, normalize
from training.sim import SKILLS, DuckEnv, hardware

In [ ]:
hw = hardware.load("2026-11-26-hd1910")
skill = SKILLS["course-walk"]
env = DuckEnv(skill, hw, num_envs=2048, device="cuda", seed=1)
env.reset()
env.randomize_episode_starts()                           # 回合进度打散，和正式训练一样
net = ActorCritic(env.obs_dim, env.critic_obs_dim, env.action_dim).to("cuda")   # 61 个观测进，14 个关节的动作出

In [ ]:
def reward_to_go(rewards, dones, gamma):
    returns = torch.zeros_like(rewards)
    running = torch.zeros_like(rewards[0])
    for t in reversed(range(len(rewards))):              # 从最后一步往前算
        running = rewards[t] + gamma * running * (1 - dones[t])   # 回合在这一步结束就乘 0
        returns[t] = running
    return returns

In [ ]:
class ReinforceData(RolloutData):                        # data 一半：每轮用当前网络采 24 步
    gamma = 0.99

    def prepare(self, rollout):
        returns = reward_to_go(rollout["rewards"], rollout["dones"], self.gamma)
        return {**flat(rollout, "obs", "actions"), "advantages": normalize(returns).flatten()}

In [ ]:
data = ReinforceData(env, net)
batch = data.prepare(data.collect())                     # 采一轮，换成训练样本
{key: tuple(value.shape) for key, value in batch.items()}

In [ ]:
class Reinforce(PolicyModule):                           # model 一半：一批数据的损失
    entropy_coef = 0.01

    def loss(self, batch):
        dist = self.net.action_distribution(batch["obs"])
        log_probs = dist.log_prob(batch["actions"]).sum(-1)       # 14 个关节的对数概率相加
        entropy = dist.entropy().sum(-1).mean()
        policy_loss = -(log_probs * batch["advantages"]).mean()     # 策略梯度的损失
        return policy_loss - self.entropy_coef * entropy               # 减一点熵，鼓励探索

In [ ]:
before = evaluate(net, skill, hw,
                  common_step_counter=env.unwrapped.common_step_counter)   # 512 只鸭子各走 20 秒
before["falls"], round(before["mean_step_reward"], 3)

In [ ]:
# 每轮重建一次数据，也就是每轮用最新的网络重新采样
trainer = L.Trainer(max_epochs=1500, reload_dataloaders_every_n_epochs=1,
                    accelerator="gpu", devices=1, logger=False, enable_checkpointing=False)
trainer.fit(Reinforce(net), data)

In [ ]:
after = evaluate(net, skill, hw,
                 common_step_counter=env.unwrapped.common_step_counter)
after["falls"], round(after["mean_step_reward"], 3)

In [ ]:
record_video(net, skill, hw, "v1_reinforce.mp4",
             common_step_counter=env.unwrapped.common_step_counter)
Video("v1_reinforce.mp4", embed=True)

In [ ]:
torch.save(net.state_dict(), "v1_reinforce.pt")